In [3]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from mailersend import MailerSendClient, EmailBuilder
from typing import Dict
import os
import asyncio
from agents.models.openai_provider import OpenAIProvider
from agents.run import RunConfig
from openai import AsyncOpenAI

In [11]:
load_dotenv(dotenv_path="/Users/jane/Desktop/claude-code-projects/agent/agents/my_2_openai/.env", override=True)
api_key=os.getenv('MAILERSEND_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

from agents import set_default_openai_client
set_default_openai_client(client)

###run_config = RunConfig(model_provider=OpenAIProvider(openai_client=client))

In [ ]:
FROM_EMAIL = "test@test-3m5jgro0kpzgdpyo.mlsender.net"
TO_EMAIL = ""     # your recipient email

def send_test_email():
    client = MailerSendClient(api_key=os.environ.get('MAILERSEND_API_KEY'))
    email = (EmailBuilder()
        .from_email(FROM_EMAIL)
        .to(TO_EMAIL)
        .subject("Test email")
        .text("This is an important test email")
        .build())
    response = client.emails.send(email)
    print(response)

send_test_email()

{'id': '69d3d1d2f2f397a5ab128dc3'}


In [13]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [14]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="openai/gpt-4o-mini"
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="openai/gpt-4o-mini"
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="openai/gpt-4o-mini"
)

In [15]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Streamline Your SOC 2 Compliance Process with AI 

Dear [Recipient's Name],

I hope this message finds you well.

I’m reaching out to introduce you to ComplAI, a next-generation SaaS tool designed to simplify and enhance your SOC 2 compliance and audit preparation process. In today’s competitive landscape, ensuring your organization meets compliance standards is not just a necessity but also a critical differentiator.

Our platform leverages cutting-edge AI technology to automate time-consuming compliance tasks, providing you with real-time insights and guidance tailored to your specific needs. With ComplAI, you can:

- **Reduce Audit Preparation Time**: Streamline documentation and reporting processes to save valuable hours.
- **Enhance Compliance Accuracy**: Minimize human error with automated documentation checks and updates.
- **Stay Informed**: Receive proactive notifications about compliance changes relevant to your industry.

Many organizations have already benefitted f

In [16]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

Subject: Simplify Your SOC 2 Compliance Journey with ComplAI

Dear [Recipient's Name],

I hope this message finds you well.

As organizations increasingly prioritize data security and compliance, navigating the complexities of SOC 2 requirements can be a daunting task. At ComplAI, we understand the challenges that come with ensuring compliance and preparing for audits. That’s why we’ve developed an AI-powered SaaS solution designed to streamline the process.

With ComplAI, you can:

- **Automate Documentation**: Reduce manual efforts by automating compliance documentation and tracking.
- **Stay Audit-Ready**: Continuously monitor and manage your compliance posture, ensuring you’re always prepared for audits.
- **Customizable Workflows**: Tailor our platform to meet your specific business needs, making it easier to adapt to changing regulations.

Our clients have experienced significant reductions in time spent on compliance efforts, allowing them to focus on what they do best—growing t

In [18]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model="gpt-4o-mini"
)

In [19]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")

Best sales email:
Subject: Don’t Let SOC2 Compliance Be a Headache – Let Us Be Your Aspirin! 💊

Hi [First Name],

I hope this email finds you well – and not buried under a mountain of compliance paperwork! If you’re feeling overwhelmed by the thought of SOC2 audits, I've got good news: I’m here like a friendly guide through the dense forest of compliance (with no bears, I promise!).

At ComplAI, our AI-powered tool helps you sail through the SOC2 compliance process faster than you can say “cloud security.” 🛥️ Think of us as your trusty sidekick, minus the cape (and the moral dilemmas). 

With ComplAI, you can:

- Automate tedious tasks that make compliance feel like a marathon, not a sprint.
- Prepare for audits with ease—our software does most of the heavy lifting, so you can focus on more important things, like wondering what to have for lunch.
- Sleep peacefully, knowing you’re SOC2 compliant without losing your sanity. (Imagine pillow endorsements, but for stress relief.)

Let’s sc

In [20]:
@function_tool
def send_email(body: str):
    """ Send out an email with the given body to all sales prospects """
    client = MailerSendClient(api_key=os.environ.get('MAILERSEND_API_KEY'))
    email = (EmailBuilder()
        .from_email(FROM_EMAIL)
        .to(TO_EMAIL)
        .subject("Sales email")
        .text(body)
        .build())
    client.emails.send(email)
    return {"status": "success"}

In [21]:
from agents import set_default_openai_client
set_default_openai_client(client)

description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.

Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
2. Evaluate and Select: Review the drafts and choose the single best email.
3. Use the send_email tool to send the best email (and only the best email) to the user.

Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="openai/gpt-4o-mini")

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)
    print(result.final_output)

The cold sales email has been successfully sent. If you need any further assistance or additional emails, feel free to ask!


In [ ]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    client = MailerSendClient(api_key=os.environ.get('MAILERSEND_API_KEY'))
    email = (EmailBuilder()
        .from_email(FROM_EMAIL)
        .to(TO_EMAIL)
        .subject(subject)
        .html(html_body)
        .build())
    client.emails.send(email)
    return {"status": "success"}

subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="openai/gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="openai/gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Convert a text email body to an HTML email body")

emailer_instructions = "You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."

emailer_agent = Agent(
    name="Email Manager",
    instructions=emailer_instructions,
    tools=[subject_tool, html_tool, send_html_email],
    model="openai/gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it")

sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.

Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
2. Evaluate and Select: Review the drafts and choose the single best email.
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent.

Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""

sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=[tool1, tool2, tool3],
    handoffs=[emailer_agent],
    model="openai/gpt-4o-mini")

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

CancelledError: 

In [ ]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")
